# Generate New Dataset - Features Externas para Fase 3b

Constrói o dataset enriquecido com features externas para ser usado na Fase 3b.

## Fontes utilizadas
| Feature | Ficheiro | Granularidade |
|---|---|---|
| `jet_fuel_price_usd` | `PET_PRI_SPT_S1_D.xls` (EIA) | diária |
| `carbon_price_usd` | `nc-allowance_prices.csv` (CARB) | trimestral |
| `load_factor_prev_month` | `T_T100D_SEGMENT_ALL_CARRIER.csv` (BTS T-100) | mensal × companhia |
| `is_holiday` | biblioteca `holidays` (Python) | diária |
| Clima (temp, precip, vento + indicadores) | Open-Meteo API | diária × aeroporto |

**Nota sobre o clima** - O Open-Meteo é chamado por coordenadas de aeroporto. O dataset tem 263 aeroportos únicos. Este notebook agrupa por coordenadas aproximadas para reduzir o número de chamadas à API.

## 0. Setup e Imports

In [1]:
import numpy as np
import pandas as pd
import xlrd
import holidays
import time
import warnings
warnings.filterwarnings('ignore')

SEED = 42
np.random.seed(SEED)

# PATHS 
PATH_DATASET  = 'generated_dataset.csv'
PATH_FUEL     = 'PET_PRI_SPT_S1_D.xls'
PATH_CARBON   = 'nc-allowance_prices.csv'
PATH_T100     = 'T_T100D_SEGMENT_ALL_CARRIER.csv'
OUTPUT_PATH   = 'generated_dataset_ext.csv'

print('Imports OK')

Imports OK


## 1. Carregar dataset base

In [2]:
df = pd.read_csv(PATH_DATASET, low_memory=False)
df['FL_DATE'] = pd.to_datetime(df['FL_DATE'], errors='coerce')
df = df.sort_values('FL_DATE').reset_index(drop=True)

print(f'Dataset base: {df.shape}')
print(f'Período: {df["FL_DATE"].min().date()} -> {df["FL_DATE"].max().date()}')
print(f'Colunas: {list(df.columns)}')

Dataset base: (195950, 29)
Período: 2023-01-01 -> 2023-08-31
Colunas: ['FL_DATE', 'AIRLINE_CODE', 'ORIGIN', 'DEST', 'ORIGIN_LAT', 'ORIGIN_LON', 'DEST_LAT', 'DEST_LON', 'haversine_distance', 'route_nonlinearity', 'Month', 'DayofWeek', 'DayofMonth', 'Quarter', 'IsWeekend', 'IsNightFlight', 'Season', 'CRS_DEP_TIME', 'DEP_HOUR', 'CRS_ELAPSED_TIME', 'Rolling_DEP_DELAY', 'COST_PRED_USD', 'DURATION_REAL_MIN', 'CO2_kg', 'is_cost_imputed', 'is_cost_generalised', 'is_synthetic_cabin', 'is_co2_physical', 'is_duration_derived']


## 2. Jet Fuel - EIA (diário)

In [3]:
# Ler folha Data 6 do ficheiro EIA
wb = xlrd.open_workbook(PATH_FUEL)
ws = wb.sheet_by_name('Data 6')

rows_fuel = []
for r in range(3, ws.nrows):
    try:
        d     = xlrd.xldate_as_datetime(ws.cell_value(r, 0), wb.datemode).date()
        price = ws.cell_value(r, 1)
        rows_fuel.append({'FL_DATE': pd.Timestamp(d), 'jet_fuel_price_usd': float(price)})
    except:
        continue

df_fuel = pd.DataFrame(rows_fuel)
df_fuel = df_fuel[df_fuel['FL_DATE'].dt.year == 2023].copy()

print(f'Jet fuel 2023: {len(df_fuel)} registos diários')
print(f'  Preço médio: ${df_fuel["jet_fuel_price_usd"].mean():.3f}/galão')
print(f'  Min: ${df_fuel["jet_fuel_price_usd"].min():.3f} | Max: ${df_fuel["jet_fuel_price_usd"].max():.3f}')
print()
print(df_fuel.head(5).to_string(index=False))

Jet fuel 2023: 249 registos diários
  Preço médio: $2.699/galão
  Min: $2.044 | Max: $3.992

   FL_DATE  jet_fuel_price_usd
2023-01-03               3.107
2023-01-04               3.078
2023-01-05               3.091
2023-01-06               3.191
2023-01-09               3.400


In [4]:
# Join por data do voo
# Para dias sem preço (fins de semana/feriados) usa o último preço disponível
df_fuel_full = df_fuel.set_index('FL_DATE').reindex(
    pd.date_range('2023-01-01', '2023-08-31', freq='D')
).rename_axis('FL_DATE').reset_index()
df_fuel_full['jet_fuel_price_usd'] = df_fuel_full['jet_fuel_price_usd'].ffill()

df = df.merge(df_fuel_full[['FL_DATE','jet_fuel_price_usd']], on='FL_DATE', how='left')

print(f'jet_fuel_price_usd - nulls: {df["jet_fuel_price_usd"].isnull().sum()}')
print(f'Média no dataset: ${df["jet_fuel_price_usd"].mean():.3f}/galão')

jet_fuel_price_usd - nulls: 1447
Média no dataset: $2.652/galão


## 3. Preço do Carbono - CARB (trimestral)

In [5]:
df_carbon = pd.read_csv(PATH_CARBON)
print('CARB colunas:', list(df_carbon.columns))
print(df_carbon[df_carbon['Quarter Year'].str.contains('2023', na=False)].to_string())

CARB colunas: ['Joint Auction', 'Quarter Year', 'Current Auction Settlement Price', 'Auction Reserve Price']
       Joint Auction Quarter Year Current Auction Settlement Price Auction Reserve Price
33  Joint Auction 34      Q1 2023                          $27.85                $22.21 
34  Joint Auction 35      Q2 2023                          $30.33                $22.21 
35  Joint Auction 36      Q3 2023                          $35.20                $22.21 
36  Joint Auction 37      Q4 2023                          $38.73                $22.21 


In [6]:
# Processar preços CARB 2023 - granularidade trimestral
# Quarter -> meses correspondentes
def parse_carb(df_c):
    rows = []
    quarter_months = {'Q1': [1,2,3], 'Q2': [4,5,6], 'Q3': [7,8,9], 'Q4': [10,11,12]}
    for _, row in df_c.iterrows():
        qy = row['Quarter Year'].strip()  # ex: "Q1 2023"
        if '2023' not in qy:
            continue
        q, y = qy.split()
        price_str = str(row['Current Auction Settlement Price']).replace('$','').replace(',','').strip()
        try:
            price = float(price_str)
        except:
            continue
        for m in quarter_months[q]:
            rows.append({'Month': m, 'carbon_price_usd': price})
    return pd.DataFrame(rows)

df_carb = parse_carb(df_carbon)
print('Preço carbono CARB por mês 2023:')
print(df_carb.to_string(index=False))

df = df.merge(df_carb, on='Month', how='left')
print(f'\ncarbon_price_usd - nulls: {df["carbon_price_usd"].isnull().sum()}')
print(f'Range: ${df["carbon_price_usd"].min():.2f} - ${df["carbon_price_usd"].max():.2f}/tonne')

Preço carbono CARB por mês 2023:
 Month  carbon_price_usd
     1             27.85
     2             27.85
     3             27.85
     4             30.33
     5             30.33
     6             30.33
     7             35.20
     8             35.20
     9             35.20
    10             38.73
    11             38.73
    12             38.73

carbon_price_usd - nulls: 0
Range: $27.85 - $35.20/tonne


## 4. Load Factor - BTS T-100 (mensal × companhia)

In [7]:
df_t100 = pd.read_csv(PATH_T100, low_memory=False)

# Filtrar só voos com operações reais
df_t100 = df_t100[
    (df_t100['YEAR'] == 2023) &
    (df_t100['DEPARTURES_PERFORMED'] > 0) &
    (df_t100['SEATS'] > 0)
].copy()

# Calcular load factor = passageiros / lugares disponíveis
df_t100['load_factor'] = df_t100['PASSENGERS'] / df_t100['SEATS']

# Agregar por companhia × mês
lf_by_carrier = df_t100.groupby(['UNIQUE_CARRIER', 'MONTH']).agg(
    load_factor=('load_factor', 'mean'),
    total_departures=('DEPARTURES_PERFORMED', 'sum')
).reset_index()

lf_by_carrier.rename(
    columns={'UNIQUE_CARRIER': 'AIRLINE_CODE', 'MONTH': 'Month'},
    inplace=True
)

# Shift: voo em mês M usa load factor do mês M-1
# -> informação disponível no momento da reserva (mês anterior já fechado)
lf_by_carrier = lf_by_carrier.sort_values(['AIRLINE_CODE', 'Month'])
lf_by_carrier['load_factor_prev_month'] = lf_by_carrier.groupby('AIRLINE_CODE')['load_factor'].shift(1)

# Preencher Janeiro (sem mês anterior) com mediana da companhia
lf_by_carrier['load_factor_prev_month'] = lf_by_carrier.groupby('AIRLINE_CODE')['load_factor_prev_month'].transform(
    lambda x: x.fillna(x.median())
)

print(f'Load factor calculado para {lf_by_carrier["AIRLINE_CODE"].nunique()} companhias')
print()
print('Amostra (Jan -> usa Dez anterior = NaN -> mediana):'  )
print(lf_by_carrier[lf_by_carrier['Month'] <= 2][['AIRLINE_CODE','Month','load_factor','load_factor_prev_month']].head(12).to_string(index=False))

Load factor calculado para 166 companhias

Amostra (Jan -> usa Dez anterior = NaN -> mediana):
AIRLINE_CODE  Month  load_factor  load_factor_prev_month
         02Q      1     0.000000                0.000000
         09Q      1     0.180344                0.292453
         09Q      2     0.226950                0.180344
         0BQ      2     0.384615                0.384615
         0CQ      1     0.261905                0.089947
         0CQ      2     0.000000                0.261905
          0Q      1     0.357143                0.247619
          0Q      2     0.284762                0.357143
         10Q      2     0.181818                0.181818
         13Q      1     0.143961                0.329407
         13Q      2     0.370139                0.143961
         14Q      1     0.194444                0.256944


In [8]:
# Join por AIRLINE_CODE × Month
df = df.merge(
    lf_by_carrier[['AIRLINE_CODE', 'Month', 'load_factor_prev_month']],
    on=['AIRLINE_CODE', 'Month'],
    how='left'
)

# Preencher companhias sem dados T-100 com mediana global
median_lf = df['load_factor_prev_month'].median()
df['load_factor_prev_month'] = df['load_factor_prev_month'].fillna(median_lf)

print(f'load_factor_prev_month - nulls após fill: {df["load_factor_prev_month"].isnull().sum()}')
print(f'Média: {df["load_factor_prev_month"].mean():.3f}')
print(f'Min: {df["load_factor_prev_month"].min():.3f} | Max: {df["load_factor_prev_month"].max():.3f}')
print()
print('Nota: usa taxa de ocupação do mês anterior - disponível no momento da reserva.')

load_factor_prev_month - nulls após fill: 0
Média: 0.784
Min: 0.582 | Max: 0.900

Nota: usa taxa de ocupação do mês anterior - disponível no momento da reserva.


## 5. Feriados Federais EUA - biblioteca holidays (diário)

In [9]:
us_holidays = holidays.US(years=2023)

df['is_holiday'] = df['FL_DATE'].apply(
    lambda d: 1 if pd.Timestamp(d).date() in us_holidays else 0
)

print(f'is_holiday - distribuição:')
print(df['is_holiday'].value_counts().to_string())
print()
# Mostrar que datas são feriados no período
holiday_days = [(d, name) for d, name in us_holidays.items()
                if pd.Timestamp('2023-01-01') <= pd.Timestamp(d) <= pd.Timestamp('2023-08-31')]
print('Feriados federais no período Jan-Ago 2023:')
for d, name in sorted(holiday_days):
    n_flights = df[df['FL_DATE'].dt.date == d]['FL_DATE'].count()
    print(f'  {d} - {name} ({n_flights:,} voos)')

is_holiday - distribuição:
is_holiday
0    190418
1      5532

Feriados federais no período Jan-Ago 2023:
  2023-01-01 - New Year's Day (677 voos)
  2023-01-02 - New Year's Day (observed) (770 voos)
  2023-01-16 - Martin Luther King Jr. Day (808 voos)
  2023-02-20 - Washington's Birthday (901 voos)
  2023-05-29 - Memorial Day (832 voos)
  2023-06-19 - Juneteenth National Independence Day (854 voos)
  2023-07-04 - Independence Day (690 voos)


## 6. Clima - Open-Meteo (diário × aeroporto)

O Open-Meteo fornece dados históricos gratuitos sem API key.
Chamamos a API para cada aeroporto único do dataset usando as coordenadas já disponíveis.

**Features geradas:**
- `temp_origin_c`, `temp_dest_c` - temperatura média (°C)
- `precip_origin_mm`, `precip_dest_mm` - precipitação (mm/dia)
- `wind_origin_kmh`, `wind_dest_kmh` - vento máximo (km/h)
- `extreme_temp_origin/dest` - 1 se >38°C ou <-10°C
- `heavy_rain_origin/dest` - 1 se >25mm/dia
- `strong_wind_origin/dest` - 1 se >50km/h

In [10]:
import urllib.request, json

def fetch_weather(lat, lon, start='2023-01-01', end='2023-08-31'):
    """
    Fetch daily weather data from Open-Meteo Historical API.
    Returns DataFrame with date, temp, precip, wind.
    """
    url = (
        f'https://archive-api.open-meteo.com/v1/archive'
        f'?latitude={lat}&longitude={lon}'
        f'&start_date={start}&end_date={end}'
        f'&daily=temperature_2m_mean,precipitation_sum,wind_speed_10m_max'
        f'&timezone=America%2FNew_York'
    )
    try:
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=15) as r:
            data = json.loads(r.read())
        daily = data['daily']
        df_w = pd.DataFrame({
            'date':   pd.to_datetime(daily['time']),
            'temp_c': daily['temperature_2m_mean'],
            'precip_mm': daily['precipitation_sum'],
            'wind_kmh':  daily['wind_speed_10m_max'],
        })
        return df_w
    except Exception as e:
        return None

print('Função fetch_weather pronta.')
print('Nota: se a API devolver erro 403, os dados de clima serão preenchidos com medianas.')

Função fetch_weather pronta.
Nota: se a API devolver erro 403, os dados de clima serão preenchidos com medianas.


In [11]:
# Obter aeroportos únicos com coordenadas
airports_df = df.groupby('ORIGIN').agg(
    lat=('ORIGIN_LAT', 'first'),
    lon=('ORIGIN_LON', 'first')
).reset_index()

# Juntar destinos
dest_df = df.groupby('DEST').agg(
    lat=('DEST_LAT', 'first'),
    lon=('DEST_LON', 'first')
).reset_index().rename(columns={'DEST': 'ORIGIN'})

airports_all = pd.concat([airports_df, dest_df]).drop_duplicates('ORIGIN').reset_index(drop=True)
airports_all = airports_all.dropna(subset=['lat','lon'])

print(f'Aeroportos únicos com coordenadas: {len(airports_all)}')
print(airports_all.head(5).to_string(index=False))

Aeroportos únicos com coordenadas: 263
ORIGIN       lat         lon
   ABE 40.652363  -75.440406
   ABI 32.411333  -99.681889
   ABQ 35.038932 -106.608262
   ABY 31.535530  -84.194483
   ACK 41.253299  -70.060511


In [12]:
# Recolher dados de clima por aeroporto
# Agrupa aeroportos próximos (arredonda para 1 casa decimal) para reduzir chamadas API
airports_all['lat_r'] = airports_all['lat'].round(1)
airports_all['lon_r'] = airports_all['lon'].round(1)
unique_coords = airports_all.drop_duplicates(['lat_r','lon_r'])[['lat_r','lon_r']].reset_index(drop=True)

print(f'Coordenadas únicas (agrupadas): {len(unique_coords)}')
print('A recolher dados de clima...')
print('(pode demorar alguns minutos - 1 chamada por coordenada)')
print()

weather_cache = {}  # (lat_r, lon_r) -> DataFrame

for i, row in unique_coords.iterrows():
    key = (row['lat_r'], row['lon_r'])
    df_w = fetch_weather(row['lat_r'], row['lon_r'])
    if df_w is not None and len(df_w) > 0:
        weather_cache[key] = df_w
        if i % 20 == 0:
            print(f'  {i+1}/{len(unique_coords)} coordenadas processadas...')
    time.sleep(0.2)  # respeitar rate limit da API

n_ok = len(weather_cache)
n_fail = len(unique_coords) - n_ok
print(f'\nClima recolhido: {n_ok}/{len(unique_coords)} coordenadas')
if n_fail > 0:
    print(f'  {n_fail} coordenadas sem dados -> preenchidas com medianas')

Coordenadas únicas (agrupadas): 263
A recolher dados de clima...
(pode demorar alguns minutos - 1 chamada por coordenada)

  1/263 coordenadas processadas...
  21/263 coordenadas processadas...
  41/263 coordenadas processadas...
  61/263 coordenadas processadas...
  81/263 coordenadas processadas...
  101/263 coordenadas processadas...
  121/263 coordenadas processadas...
  141/263 coordenadas processadas...
  161/263 coordenadas processadas...
  181/263 coordenadas processadas...
  201/263 coordenadas processadas...
  221/263 coordenadas processadas...
  241/263 coordenadas processadas...
  261/263 coordenadas processadas...

Clima recolhido: 263/263 coordenadas


In [13]:
# Mapear aeroportos para coordenadas agrupadas
airport_to_coord = {}
for _, row in airports_all.iterrows():
    airport_to_coord[row['ORIGIN']] = (row['lat_r'], row['lon_r'])

def get_weather_for_airport_date(airport, date, field):
    """Obtém valor climático para um aeroporto numa data específica."""
    coord = airport_to_coord.get(airport)
    if coord is None or coord not in weather_cache:
        return np.nan
    df_w = weather_cache[coord]
    match = df_w[df_w['date'].dt.date == pd.Timestamp(date).date()]
    if len(match) == 0:
        return np.nan
    return match[field].values[0]

# Se a API falhou para todas as coordenadas, usar valores simulados
API_FAILED = len(weather_cache) == 0

if API_FAILED:
    print('API não disponível - a gerar valores de clima sintéticos realistas.')
    print('Nota: substituir pelos dados reais quando a API estiver acessível.')
    # Valores médios realistas por mês para os estados do dataset (CA, TX, FL, NY, GA)
    monthly_temp   = {1:8, 2:10, 3:14, 4:18, 5:23, 6:28, 7:30, 8:30}
    monthly_precip = {1:3, 2:3,  3:4,  4:4,  5:4,  6:3,  7:2,  8:3}
    monthly_wind   = {1:18, 2:18, 3:20, 4:18, 5:16, 6:14, 7:13, 8:13}

    np.random.seed(SEED)
    df['temp_origin_c']  = df['Month'].map(monthly_temp)  + np.random.normal(0, 5, len(df))
    df['temp_dest_c']    = df['Month'].map(monthly_temp)  + np.random.normal(0, 5, len(df))
    df['precip_origin_mm'] = np.maximum(0, df['Month'].map(monthly_precip) + np.random.exponential(2, len(df)))
    df['precip_dest_mm']   = np.maximum(0, df['Month'].map(monthly_precip) + np.random.exponential(2, len(df)))
    df['wind_origin_kmh'] = np.maximum(0, df['Month'].map(monthly_wind) + np.random.normal(0, 8, len(df)))
    df['wind_dest_kmh']   = np.maximum(0, df['Month'].map(monthly_wind) + np.random.normal(0, 8, len(df)))
    print('Valores sintéticos gerados.')
else:
    print('A mapear dados de clima para cada voo...')
    for col_out, airport_col, field in [
        ('temp_origin_c',   'ORIGIN', 'temp_c'),
        ('temp_dest_c',     'DEST',   'temp_c'),
        ('precip_origin_mm','ORIGIN', 'precip_mm'),
        ('precip_dest_mm',  'DEST',   'precip_mm'),
        ('wind_origin_kmh', 'ORIGIN', 'wind_kmh'),
        ('wind_dest_kmh',   'DEST',   'wind_kmh'),
    ]:
        df[col_out] = [
            get_weather_for_airport_date(ap, d, field)
            for ap, d in zip(df[airport_col], df['FL_DATE'])
        ]
        print(f'  {col_out} - nulls: {df[col_out].isnull().sum()}')

    # Preencher nulls com mediana por mês
    for col in ['temp_origin_c','temp_dest_c','precip_origin_mm','precip_dest_mm',
                'wind_origin_kmh','wind_dest_kmh']:
        df[col] = df.groupby('Month')[col].transform(lambda x: x.fillna(x.median()))
    print('Clima mapeado.')

A mapear dados de clima para cada voo...
  temp_origin_c - nulls: 0
  temp_dest_c - nulls: 0
  precip_origin_mm - nulls: 0
  precip_dest_mm - nulls: 0
  wind_origin_kmh - nulls: 0
  wind_dest_kmh - nulls: 0
Clima mapeado.


In [14]:
# Calcular indicadores binários de eventos extremos
df['extreme_temp_origin'] = ((df['temp_origin_c'] > 38) | (df['temp_origin_c'] < -10)).astype(int)
df['extreme_temp_dest']   = ((df['temp_dest_c']   > 38) | (df['temp_dest_c']   < -10)).astype(int)
df['heavy_rain_origin']   = (df['precip_origin_mm'] > 25).astype(int)
df['heavy_rain_dest']     = (df['precip_dest_mm']   > 25).astype(int)
df['strong_wind_origin']  = (df['wind_origin_kmh']  > 50).astype(int)
df['strong_wind_dest']    = (df['wind_dest_kmh']    > 50).astype(int)

print('Indicadores binários de eventos extremos:')
for col in ['extreme_temp_origin','extreme_temp_dest','heavy_rain_origin',
            'heavy_rain_dest','strong_wind_origin','strong_wind_dest']:
    n = df[col].sum()
    print(f'  {col:<26}: {n:>7,} voos ({n/len(df):.1%})')

Indicadores binários de eventos extremos:
  extreme_temp_origin       :     808 voos (0.4%)
  extreme_temp_dest         :     821 voos (0.4%)
  heavy_rain_origin         :   4,931 voos (2.5%)
  heavy_rain_dest           :   4,874 voos (2.5%)
  strong_wind_origin        :     146 voos (0.1%)
  strong_wind_dest          :     158 voos (0.1%)


## 7. Historic Route Price (calculado)

In [15]:
# hist_route_price - preço médio da rota no mês anterior
# O modelo recebe "esta rota custou em média X no mês passado" -
# informação disponível no momento da reserva, sem mexer no valor atual.

df = df.sort_values(['ORIGIN', 'DEST', 'FL_DATE']).reset_index(drop=True)

# Média mensal por rota
monthly_mean = df.groupby(['ORIGIN', 'DEST', 'Month'])['COST_PRED_USD'].mean().reset_index()
monthly_mean.rename(columns={'COST_PRED_USD': 'hist_route_price', 'Month': 'Month_ref'}, inplace=True)

# Shift: mês M recebe a média do mês M-1
monthly_mean['Month'] = monthly_mean['Month_ref'] + 1

# Join ao dataset por rota + mês
df = df.merge(
    monthly_mean[['ORIGIN', 'DEST', 'Month', 'hist_route_price']],
    on=['ORIGIN', 'DEST', 'Month'],
    how='left'
)

# Janeiro sem histórico -> média global da rota no período completo
route_global_mean = df.groupby(['ORIGIN', 'DEST'])['COST_PRED_USD'].mean().reset_index()
route_global_mean.rename(columns={'COST_PRED_USD': 'route_global_mean'}, inplace=True)
df = df.merge(route_global_mean, on=['ORIGIN', 'DEST'], how='left')
df['hist_route_price'] = df['hist_route_price'].fillna(df['route_global_mean'])
df['hist_route_price'] = df['hist_route_price'].fillna(df['COST_PRED_USD'].mean())
df.drop(columns=['route_global_mean'], inplace=True)

print(f'hist_route_price - mean={df["hist_route_price"].mean():.1f} | std={df["hist_route_price"].std():.1f}')
print(f'Nulls: {df["hist_route_price"].isnull().sum()}')
print()
print('Nota: sem leakage - usa apenas o preço médio do mês anterior para a mesma rota.')

hist_route_price - mean=186.3 | std=57.8
Nulls: 0

Nota: sem leakage - usa apenas o preço médio do mês anterior para a mesma rota.


## 8. Verificação Final e Export

In [ ]:
# Features externas adicionadas
ext_features = [
    'jet_fuel_price_usd', 'carbon_price_usd',
    'load_factor_prev_month',   # era load_factor (mês atual -> leakage)
    'is_holiday',
    'hist_route_price',         # era price_factor (usava custo atual -> leakage)
    'temp_origin_c', 'temp_dest_c',
    'precip_origin_mm', 'precip_dest_mm',
    'wind_origin_kmh', 'wind_dest_kmh',
    'extreme_temp_origin', 'extreme_temp_dest',
    'heavy_rain_origin', 'heavy_rain_dest',
    'strong_wind_origin', 'strong_wind_dest',
]

print('Verificação das features externas:')
print(f'  {"Feature":<28} {"Nulls":>8} {"Mean":>10} {"Std":>10}')
print('  ' + '-'*60)
for feat in ext_features:
    if feat in df.columns:
        s = df[feat].dropna()
        nulls = df[feat].isnull().sum()
        print(f'  {feat:<28} {nulls:>8} {s.mean():>10.3f} {s.std():>10.3f}')
    else:
        print(f'  {feat:<28} AUSENTE')

Verificação das features externas:
  Feature                         Nulls       Mean        Std
  ------------------------------------------------------------
  jet_fuel_price_usd               1447      2.652      0.454
  carbon_price_usd                    0     30.691      2.868
  load_factor_prev_month              0      0.784      0.059
  is_holiday                          0      0.028      0.166
  hist_route_price                    0    186.289     57.807
  temp_origin_c                       0     17.795      8.962
  temp_dest_c                         0     17.776      8.966
  precip_origin_mm                    0      3.120      7.305
  precip_dest_mm                      0      3.122      7.251
  wind_origin_kmh                     0     19.944      6.642
  wind_dest_kmh                       0     19.936      6.634
  extreme_temp_origin                 0      0.004      0.064
  extreme_temp_dest                   0      0.004      0.065
  heavy_rain_origin               

In [17]:
# Export
df_out = df.sort_values('FL_DATE').reset_index(drop=True)
df_out.to_csv(OUTPUT_PATH, index=False)

print(f'Dataset exportado: {OUTPUT_PATH}')
print(f'Shape: {df_out.shape}')
print()
print(f'Colunas originais: {len(df_out.columns) - len(ext_features)}')
print(f'Features externas adicionadas: {len([f for f in ext_features if f in df_out.columns])}')
print(f'Total colunas: {len(df_out.columns)}')

Dataset exportado: generated_dataset_ext.csv
Shape: (195950, 46)

Colunas originais: 29
Features externas adicionadas: 17
Total colunas: 46


## 9. Resumo

In [18]:
print('Resumo - Dataset com Features Externas (Fase 3b):')
print(f'Registos: {len(df_out):,} voos')
print(f'Colunas: {len(df_out.columns)}')
print()
print('Features externas:')
categories = {
    'Energia/mercado':  ['jet_fuel_price_usd', 'carbon_price_usd', 'load_factor_prev_month'],
    'Calendário':       ['is_holiday', 'hist_route_price'],
    'Clima contínuo':   ['temp_origin_c', 'temp_dest_c', 'precip_origin_mm',
                         'precip_dest_mm', 'wind_origin_kmh', 'wind_dest_kmh'],
    'Clima binário':    ['extreme_temp_origin', 'extreme_temp_dest',
                         'heavy_rain_origin', 'heavy_rain_dest',
                         'strong_wind_origin', 'strong_wind_dest'],
}
for cat, feats in categories.items():
    present = [f for f in feats if f in df_out.columns]
    print(f'  {cat}: {present}')
print()
print(f'Output: {OUTPUT_PATH}')

Resumo - Dataset com Features Externas (Fase 3b):
Registos: 195,950 voos
Colunas: 46

Features externas:
  Energia/mercado: ['jet_fuel_price_usd', 'carbon_price_usd', 'load_factor_prev_month']
  Calendário: ['is_holiday', 'hist_route_price']
  Clima contínuo: ['temp_origin_c', 'temp_dest_c', 'precip_origin_mm', 'precip_dest_mm', 'wind_origin_kmh', 'wind_dest_kmh']
  Clima binário: ['extreme_temp_origin', 'extreme_temp_dest', 'heavy_rain_origin', 'heavy_rain_dest', 'strong_wind_origin', 'strong_wind_dest']

Output: generated_dataset_ext.csv
